In [ ]:
#STEP 1 Authentication
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
import pandas as pd
import numpy as np
from itertools import combinations
import matplotlib.pyplot as plt
import seaborn as sns

# Koneksi ke Google Sheets
creds, _ = default()
gc = gspread.authorize(creds)

print("Step 1 Berhasil: Library dimuat dan akses Google diberikan.")

In [ ]:
import gspread
from google.auth import default
import pandas as pd
import numpy as np
from itertools import combinations
import matplotlib.pyplot as plt
import seaborn as sns

#STEP 2 Data Gathering
# Nama tab yang ingin diambil
nama_tab = "all_ecomm_update"
gsheet_url = 'https://docs.google.com/spreadsheets/d/1kPTPLH349OZxWV7wTZQ-mjSZIJfHUt-V8blOT5O_w3A/edit?gid=0#gid=0'

try:
    sh = gc.open_by_url(gsheet_url)
    worksheet = sh.worksheet(nama_tab)
    data = worksheet.get_all_values()

    df_raw = pd.DataFrame(data[1:], columns=data[0])

    # Tambahkan filter untuk category_sku jika kolomnya ada
    if 'category_sku' in df_raw.columns:
        df_raw = df_raw[df_raw['category_sku'] == 'Single Product']
        print("Filter 'category_sku = Single Product' telah diterapkan.")
    else:
        print("Peringatan: Kolom 'category_sku' tidak ditemukan. Filter tidak diterapkan.")

    # Mengambil kolom D (no_pesanan) dan J (product_component)
    df = df_raw[['no_pesanan', 'product_component']].copy()

    # --- MEMBERSIHKAN DATA KOSONG ---
    # Hapus baris jika no_pesanan kosong ATAU product_component kosong/hanya spasi
    df = df[df['no_pesanan'].str.strip() != ""]
    df = df[df['product_component'].str.strip() != ""]

    df = df.reset_index(drop=True)

    print(f"Step 2 Berhasil: {len(df)} baris data valid berhasil ditarik dari tab '{nama_tab}'.")
    display(df.head())

except Exception as e:
    print(f"Error di Step 2: {e}")

In [ ]:
#STEP 3 Confidence & Lift Metrics Calculation
def calculate_metrics(df):
    # Hitung total transaksi unik dalam dataset
    total_tx = df['no_pesanan'].nunique()

    # Hitung berapa kali setiap produk muncul secara unik per transaksi
    product_counts = df.groupby('product_component')['no_pesanan'].nunique()

    # Kelompokkan produk berdasarkan nomor pesanan
    tx_groups = df.groupby('no_pesanan')['product_component'].apply(set)

    pair_counts = {}
    for products in tx_groups:
        # Hanya hitung jika ada lebih dari satu jenis produk dalam satu pesanan
        if len(products) > 1:
            for pair in combinations(sorted(products), 2):
                pair_counts[pair] = pair_counts.get(pair, 0) + 1

    results = []
    for (item_a, item_b), count in pair_counts.items():
        # Hitung Probabilitas individual (Support)
        sup_a = product_counts[item_a] / total_tx
        sup_b = product_counts[item_b] / total_tx

        # Hitung Lift (L-Score): Menunjukkan kekuatan hubungan
        lift = (count / total_tx) / (sup_a * sup_b)

        # Hitung Confidence (C-Score): A -> B dan B -> A
        # Data disimpan dengan presisi penuh untuk keperluan tie-breaking di Step 4
        results.append({
            'Product A': item_a,
            'Product B': item_b,
            'C_Score': count / product_counts[item_a],
            'L_Score': lift
        })
        results.append({
            'Product A': item_b,
            'Product B': item_a,
            'C_Score': count / product_counts[item_b],
            'L_Score': lift
        })

    return pd.DataFrame(results)

# Eksekusi perhitungan metrics
metrics_df = calculate_metrics(df)

if not metrics_df.empty:
    print("Step 3 Berhasil: Metrics Confidence dan Lift telah dihitung dengan presisi tinggi.")
else:
    print("Peringatan: Tidak ditemukan pasangan produk dalam transaksi yang sama.")

In [ ]:
#STEP 4 Ranking Calculation
if not metrics_df.empty:
    final_list = []

    # Lakukan perankingan untuk setiap 'Product A' secara individual
    for product in metrics_df['Product A'].unique():
        subset = metrics_df[metrics_df['Product A'] == product].copy()

        # R-Rank dasar menggunakan metode 'min' (standar statistik)
        subset['C-Rank'] = subset['C_Score'].rank(ascending=False, method='min')
        subset['L-Rank'] = subset['L_Score'].rank(ascending=False, method='min')

        # Hitung ΣR (Sum of Rank)
        subset['Sum_R_Raw'] = subset['C-Rank'] + subset['L-Rank']

        # LOGIKA TIE-BREAKER: Mengurutkan ΣR terkecil, lalu C_Score tertinggi, lalu L_Score tertinggi
        subset = subset.sort_values(
            by=['Sum_R_Raw', 'C_Score', 'L_Score'],
            ascending=[True, False, False]
        )

        # Membuat ΣR-Rank yang benar-benar unik (1, 2, 3...) untuk setiap Product A
        subset['ΣR-Rank'] = range(1, len(subset) + 1)

        final_list.append(subset)

    # Gabungkan semua hasil kembali menjadi satu DataFrame
    all_results = pd.concat(final_list)
    print("Step 4 Berhasil: Peringkat afinitas unik (ΣR-Rank) telah dihitung tanpa ada ranking ganda.")
else:
    print("Gagal: Data metrics kosong, tidak bisa melanjutkan ke pemeringkatan.")

In [ ]:
# --- STEP 5: VISUALISASI DENGAN PERINGKAT 1 DI PALING ATAS ---

# Produk yang ingin dicari
pilih_produk = "KLAR Intense White PAP+ Mini Size"

if pilih_produk in all_results['Product A'].values:
    # 1. Filter data untuk produk pilihan
    output = all_results[all_results['Product A'] == pilih_produk].copy()

    # 2. LOGIKA UNIQUE RANKING (Menghindari Double Ranking)
    # Kita mengurutkan berdasarkan prioritas:
    # - Total Peringkat (Sum_R) terkecil (Utama)
    # - Confidence tertinggi (Pemecah seri 1)
    # - Lift tertinggi (Pemecah seri 2)
    output['Sum_R_Raw'] = output['C-Rank'] + output['L-Rank']
    output = output.sort_values(
        by=['Sum_R_Raw', 'C_Score', 'L_Score'],
        ascending=[True, False, False]
    )

    # Membuat peringkat unik dari 1 - 10
    output['Unique_Rank'] = range(1, len(output) + 1)
    output_top10 = output.head(10).copy()

    # 3. FORMAT TAMPILAN TABEL (Presisi 4 Desimal)
    output_display = output_top10.copy()
    output_display['Confidence'] = output_display['C_Score'].map(lambda x: '{:.4%}'.format(x))
    output_display['Lift'] = output_display['L_Score'].map(lambda x: '{:.4f}'.format(x))

    print(f"\nTop 10 Rekomendasi Bundle (Peringkat 1 Teratas): {pilih_produk}")
    display(output_display[['Product B', 'Confidence', 'C-Rank', 'Lift', 'L-Rank', 'Unique_Rank']])

    # 4. VISUALISASI GRAFIK
    plt.figure(figsize=(12, 8))

    # Agar Peringkat 1 berada di PALING ATAS grafik:
    # Seaborn memplot baris pertama (indeks 0) di posisi teratas sumbu Y.
    # Karena output_top10 sudah diurutkan (1 ke 10), kita langsung gunakan data tersebut.
    viz_data = output_top10

    # Menggunakan palet warna terbalik agar Rank 1 lebih kontras
    colors = sns.color_palette("viridis_r", len(viz_data))
    ax = sns.barplot(x='L_Score', y='Product B', data=viz_data, palette=colors)

    plt.title(f'Analisis Afinitas: {pilih_produk}\n(Peringkat 1 di Posisi Teratas)', fontsize=14)
    plt.xlabel('Lift Score (Presisi 4 Desimal)')
    plt.ylabel('Produk Rekomendasi')

    # Menambahkan anotasi Rank dan Confidence di samping setiap bar
    for i, p in enumerate(ax.patches):
        row = viz_data.iloc[i]
        rank_val = int(row['Unique_Rank'])
        conf_val = '{:.4%}'.format(row['C_Score'])
        ax.annotate(f" Rank: {rank_val} (Conf: {conf_val})",
                    (p.get_width(), p.get_y() + p.get_height()/2),
                    ha='left', va='center', xytext=(5, 0),
                    textcoords='offset points', fontweight='bold', fontsize=10)

    plt.tight_layout()
    plt.show()
else:
    print(f"Produk '{pilih_produk}' tidak ditemukan dalam analisis.")